## Init

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/axl.dxn@gmail.com/atlikon_pipeline/1_setup/utilities

## Configure widgets

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "orders", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

print(f"catalog: {catalog}, data_source: {data_source}")

## Define the tables

In [0]:
bronze_table = f"{catalog}.{bronze_schema}.{data_source}"
silver_table = f"{catalog}.{silver_schema}.{data_source}"

## Read from orders bronze staging table


In [0]:
df_orders = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.staging_{data_source};")

display(df_orders.limit(10))

## Keep only rows where order_qty is present

In [0]:
df_orders = (
    df_orders
    .filter(F.col("order_qty").isNotNull())
)

## Clean customer_id: keep numeric, else set to 999999

In [0]:
df_orders = (
    df_orders
    .withColumn(
        "customer_id",
        F.when(F.col("customer_id").rlike("^[0-9]+$"), F.col("customer_id"))
         .otherwise("999999")
         .cast("string")
    )
)

## Remove weekday name from the date text 
"Tuesday, July 01, 2025" → "July 01, 2025"

In [0]:
df_orders = (
    df_orders
    .withColumn(
        "order_placement_date",
        F.regexp_replace(F.col("order_placement_date"), r"^[A-Za-z]+,\s*", "")
    )
)

## Parse order_placement_date using multiple possible formats

In [0]:
df_orders = (
    df_orders
    .withColumn(
        "order_placement_date",
        F.coalesce(
            F.try_to_date("order_placement_date", "yyyy/MM/dd"),
            F.try_to_date("order_placement_date", "dd-MM-yyyy"),
            F.try_to_date("order_placement_date", "dd/MM/yyyy"),
            F.try_to_date("order_placement_date", "MMMM dd, yyyy"),
        )
    )
)


## Drop duplicates

In [0]:
df_orders = df_orders.dropDuplicates(["order_id", "order_placement_date", "customer_id", "product_id", "order_qty"])

## Cast product_id to string

In [0]:
df_orders = df_orders.withColumn('product_id', F.col('product_id').cast('string'))

## Join with products

In [0]:
df_products = spark.table("fmcg.silver.products")
df_joined = (
    df_orders
    .join(
        df_products, 
        on="product_id", 
        how="inner"
    )
    .select(df_orders["*"], df_products["product_code"])
)

display(df_joined.limit(10))

## Upsert operation (no staging table)

In [0]:
if not (spark.catalog.tableExists(silver_table)):
    (
        df_joined
        .write
        .format("delta")
        .option("delta.enableChangeDataFeed", "true")
        .option("mergeSchema", "true")
        .mode("overwrite")
        .saveAsTable(silver_table)
    )
else:
    silver_delta = DeltaTable.forName(spark, silver_table)

    silver_delta.alias("silver").merge(
        source = df_joined.alias("bronze"), 
        condition = "silver.order_placement_date = bronze.order_placement_date AND silver.order_id = bronze.order_id AND silver.product_code = bronze.product_code AND silver.customer_id = bronze.customer_id"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

## Staging table to process just the arrived incremental data

In [0]:
(
    df_joined.write
    .format("delta")
    .option("delta.enableChangeDataFeed", "true")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.{silver_schema}.staging_{data_source}")
)